# DDRI 정본 생성 실행 노트북

이 노트북은 `05_prediction_canonical`의 대표 실행 문서다.

원칙:
- 스크립트를 한 번에 호출하는 래퍼로 쓰지 않는다.
- 단계별로 직접 실행한다.
- 각 단계마다 중간 결과를 표로 확인한다.
- 임포트는 항상 별도 셀로 분리한다.

현재 입력 기준:
- `rep15`: `OLD_DATA/raw_data` 스냅샷
- `full161`: `OLD_DATA/full_data` 스냅샷
- `rep15`는 `2022-12` warm-up 대여이력 사용


## 1. 임포트

모든 실행 노트북은 임포트만 담당하는 셀을 따로 둔다.

이 셀에서는 경로, 모듈 로드, 데이터프레임 처리를 위한 라이브러리만 불러온다.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd


## 2. 모듈과 대상 데이터셋 선택

이 노트북은 빌더 스크립트의 함수를 직접 불러와 단계별로 실행한다.

즉, 배치용 스크립트는 함수 저장소로만 쓰고,
실제 생성 과정은 이 노트북 안에서 확인한다.


In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
SCRIPT_PATH = ROOT / 'works' / '05_prediction_canonical' / '05_scripts' / '02_ddri_prediction_canonical_builder.py'

spec = importlib.util.spec_from_file_location('canonical_builder', SCRIPT_PATH)
canonical_builder = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = canonical_builder
spec.loader.exec_module(canonical_builder)

dataset_name = 'rep15'  # 'rep15' 또는 'full161'
dataset_spec = next(spec for spec in canonical_builder.SPECS if spec.name == dataset_name)

print('dataset:', dataset_spec.name)
print('train_source:', dataset_spec.train_source)
print('test_source:', dataset_spec.test_source)
print('output_dir:', dataset_spec.output_dir)
print('warmup_source:', dataset_spec.warmup_event_source)


dataset: rep15
train_source: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_DATA/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_train_2023_2024.csv
test_source: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_DATA/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_test_2025.csv
output_dir: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/canonical_data
warmup_source: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/2022년 12월 강남구 따릉이 이용정보/서울특별시 공공자전거 대여이력 정보_2212.csv


## 3. 입력 존재 여부 확인

정본 생성은 입력 스냅샷이 실제로 존재해야만 재현 가능하다.

이 셀에서는 현재 노트북이 가리키는 입력 경로가 모두 살아 있는지 먼저 본다.


In [3]:
input_status = pd.DataFrame([
    {
        'name': 'train_source',
        'path': str(dataset_spec.train_source),
        'exists': dataset_spec.train_source.exists(),
        'size_bytes': dataset_spec.train_source.stat().st_size if dataset_spec.train_source.exists() else None,
    },
    {
        'name': 'test_source',
        'path': str(dataset_spec.test_source),
        'exists': dataset_spec.test_source.exists(),
        'size_bytes': dataset_spec.test_source.stat().st_size if dataset_spec.test_source.exists() else None,
    },
    {
        'name': 'warmup_source',
        'path': str(dataset_spec.warmup_event_source) if dataset_spec.warmup_event_source else '',
        'exists': dataset_spec.warmup_event_source.exists() if dataset_spec.warmup_event_source else False,
        'size_bytes': dataset_spec.warmup_event_source.stat().st_size if dataset_spec.warmup_event_source and dataset_spec.warmup_event_source.exists() else None,
    },
])
input_status


,name,path,exists,size_bytes
0,train_source,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_D...,True,28691225
1,test_source,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_D...,True,14315027
2,warmup_source,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/2022년...,True,317216449


## 4. train / test 입력 로드 및 기본 검증

이 단계에서 확인할 것:
- 필수 컬럼이 있는가
- `station_id + date + hour` 중복이 없는가
- 행 수와 날짜 범위가 예상과 맞는가


In [4]:
train_df = pd.read_csv(dataset_spec.train_source)
test_df = pd.read_csv(dataset_spec.test_source)

canonical_builder.validate_input(train_df, dataset_spec.train_source)
canonical_builder.validate_input(test_df, dataset_spec.test_source)

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

pd.DataFrame([
    {
        'frame': 'train',
        'rows': len(train_df),
        'stations': train_df['station_id'].nunique(),
        'date_min': train_df['date'].min(),
        'date_max': train_df['date'].max(),
        'ncols': len(train_df.columns),
    },
    {
        'frame': 'test',
        'rows': len(test_df),
        'stations': test_df['station_id'].nunique(),
        'date_min': test_df['date'].min(),
        'date_max': test_df['date'].max(),
        'ncols': len(test_df.columns),
    },
])


,frame,rows,stations,date_min,date_max,ncols
0,train,263160,15,2023-01-01,2024-12-31,15
1,test,131400,15,2025-01-01,2025-12-31,15


## 5. warm-up 생성 확인

`rep15`는 2022-12 대여이력을 계산용으로만 앞에 붙인다.

확인 포인트:
- `rep15`에서는 warm-up이 만들어져야 함
- `full161`에서는 현재 빈 프레임일 수 있음
- 최종 출력에는 이 행을 남기지 않음


In [5]:
warmup_df = canonical_builder.build_rep15_warmup_frame(dataset_spec, train_df)

if not warmup_df.empty:
    warmup_df['date'] = pd.to_datetime(warmup_df['date'])

pd.DataFrame([
    {
        'warmup_used': not warmup_df.empty,
        'warmup_rows': len(warmup_df),
        'warmup_stations': warmup_df['station_id'].nunique() if not warmup_df.empty else 0,
        'warmup_date_min': warmup_df['date'].min() if not warmup_df.empty else None,
        'warmup_date_max': warmup_df['date'].max() if not warmup_df.empty else None,
    }
])


,warmup_used,warmup_rows,warmup_stations,warmup_date_min,warmup_date_max
0,True,11160,15,2022-12-01,2022-12-31


## 6. 연속 히스토리 구성

파생은 분할별로 따로 만들지 않고,
warm-up + train + test를 먼저 이어 붙인 연속 시계열에서 만든다.


In [6]:
if not warmup_df.empty:
    history_df = pd.concat([warmup_df, train_df, test_df], ignore_index=True)
else:
    history_df = pd.concat([train_df, test_df], ignore_index=True)

history_df = history_df.sort_values(canonical_builder.KEY_COLS).reset_index(drop=True)
pd.DataFrame([
    {
        'history_rows': len(history_df),
        'history_stations': history_df['station_id'].nunique(),
        'history_date_min': history_df['date'].min(),
        'history_date_max': history_df['date'].max(),
    }
])


,history_rows,history_stations,history_date_min,history_date_max
0,405720,15,2022-12-01,2025-12-31


## 7. seasonal_mean_2023 생성

시간패턴 평균은 `2023` 구간만으로 만든다.


In [7]:
seasonal_map = canonical_builder.build_seasonal_map(train_df)
seasonal_map.head()


,station_id,weekday,hour,seasonal_mean_2023
0,2312,0,0,0.096154
1,2312,0,1,-0.096154
2,2312,0,2,-0.096154
3,2312,0,3,0.038462
4,2312,0,4,-0.057692


## 8. 정본 파생 컬럼 생성

이 단계에서 핵심 파생 컬럼과 lag / rolling / trend를 한 번에 생성한다.


In [8]:
history_out = canonical_builder.apply_canonical_transform(history_df, seasonal_map)
history_out.head()


,station_id,station_name,station_group,date,hour,rental_count,cluster,mapped_dong_code,weekday,month,...,bike_change_deseasonalized,bike_change_lag_1,bike_change_lag_24,bike_change_lag_168,bike_change_rollmean_24,bike_change_rollstd_24,bike_change_rollmean_168,bike_change_rollstd_168,bike_change_trend_1_24,bike_change_trend_24_168
0,2312,청담역 13번 출구 앞,주거 도착형,2022-12-01,0,0,2,11680565,3,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2312,청담역 13번 출구 앞,주거 도착형,2022-12-01,1,2,2,11680565,3,12,...,2.230769,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2312,청담역 13번 출구 앞,주거 도착형,2022-12-01,2,0,2,11680565,3,12,...,-1.903846,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2312,청담역 13번 출구 앞,주거 도착형,2022-12-01,3,0,2,11680565,3,12,...,0.096154,-2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2312,청담역 13번 출구 앞,주거 도착형,2022-12-01,4,0,2,11680565,3,12,...,0.115385,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. 기간별 재분할 및 검증

연속 시계열에서 파생을 다 만든 뒤,
최종 저장 직전에만 `2023+2024`와 `2025`로 다시 나눈다.


In [9]:
train_out = history_out[(history_out['date'] >= pd.Timestamp('2023-01-01')) & (history_out['date'] <= pd.Timestamp('2024-12-31'))].copy()
test_out = history_out[history_out['date'] >= pd.Timestamp('2025-01-01')].copy()

pd.DataFrame([
    {
        'frame': 'train_out',
        'rows': len(train_out),
        'stations': train_out['station_id'].nunique(),
        'bike_change_nulls': int(train_out['bike_change_raw'].isna().sum()),
    },
    {
        'frame': 'test_out',
        'rows': len(test_out),
        'stations': test_out['station_id'].nunique(),
        'bike_change_nulls': int(test_out['bike_change_raw'].isna().sum()),
    },
])


,frame,rows,stations,bike_change_nulls
0,train_out,263160,15,0
1,test_out,131400,15,0


## 10. 최종 저장

이 셀을 실행하면 현재 선택한 데이터셋의 `canonical_data`를 다시 저장한다.


In [10]:
train_save = train_out.sort_values(canonical_builder.KEY_COLS).reset_index(drop=True).copy()
test_save = test_out.sort_values(canonical_builder.KEY_COLS).reset_index(drop=True).copy()
train_save['date'] = train_save['date'].dt.strftime('%Y-%m-%d')
test_save['date'] = test_save['date'].dt.strftime('%Y-%m-%d')

train_path = dataset_spec.output_dir / 'ddri_prediction_canonical_train_2023_2024.csv'
test_path = dataset_spec.output_dir / 'ddri_prediction_canonical_test_2025.csv'
meta_path = dataset_spec.output_dir / 'ddri_prediction_canonical_meta.json'

dataset_spec.output_dir.mkdir(parents=True, exist_ok=True)
train_save.to_csv(train_path, index=False, encoding='utf-8-sig')
test_save.to_csv(test_path, index=False, encoding='utf-8-sig')

meta = {
    'name': dataset_spec.name,
    'train_source': str(dataset_spec.train_source),
    'test_source': str(dataset_spec.test_source),
    'train_output': str(train_path),
    'test_output': str(test_path),
    'warmup_used': bool(not warmup_df.empty),
    'warmup_rows': int(len(warmup_df)),
    'train_rows_out': int(len(train_save)),
    'test_rows_out': int(len(test_save)),
    'seasonal_group_count': int(len(seasonal_map)),
}
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([meta])


,name,train_source,test_source,train_output,test_output,warmup_used,warmup_rows,train_rows_out,test_rows_out,seasonal_group_count
0,rep15,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_D...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/OLD_D...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,True,11160,263160,131400,2520
